# Validation métier du RAG et des agents de Folio

Ce notebook valide la chaîne réellement utilisée dans **Archive vivante** sur les rapports présents dans le projet : Meta, Google / Alphabet, Apple et Amazon.

**Objectifs** : vérifier la recherche automatique, les documents et exercices retenus, la présence des chiffres attendus, les citations, l’abstention et les limites des agents. Le mode live ajoute les réponses Gemini et la synthèse thématique.

Les conversations de test sont isolées et supprimées à la fin de chaque exécution. Aucun PDF, historique utilisateur ou secret n’est modifié. Le notebook n’affiche ni clé, ni mot de passe, ni raisonnement privé.

## 1. Ce qui est identique à l’application

| Étape | Code exécuté |
| --- | --- |
| Catalogue et sessions | `interface/server/app-store.mjs` |
| Recherche dans les textes extraits | `retrieval.mjs` et `rag-preparation.mjs` |
| Budgets, permissions, contrôle des références | `agent-policies.mjs` et `ai-connector.mjs` |
| Transport vers le modèle | `gemini-provider.mjs` |
| Chat et graphe planificateur → chercheur → vérificateur → rédacteur | `scripts/folio_worker.py` |

Le petit lanceur `interface/scripts/validate-business-rag.mjs` importe ces modules ; il ne réimplémente pas le RAG en Python. La recherche actuelle est **BM25 locale**, avec filtres entreprise, année et type de rapport. **Qdrant n’est pas utilisé**. Le modèle configuré est lu comme dans l’application ; sa valeur n’est pas codée dans ce notebook.

Le lanceur travaille directement dans la couche métier locale. Il n’ouvre pas de route HTTP sans authentification : les API du site restent protégées par la connexion.

In [1]:
from pathlib import Path
import os, json, shutil, subprocess, html
from datetime import datetime, timezone
from IPython.display import display, Markdown, HTML

def find_project_root():
    configured = os.getenv("FOLIO_PROJECT_ROOT")
    candidates = [Path(configured).expanduser()] if configured else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "interface/server/ai-connector.mjs").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Ouvrez le notebook depuis le dépôt Folio ou définissez FOLIO_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
NODE = shutil.which("node")
assert NODE, "Node.js 22.22+ doit être installé."
RUNNER = PROJECT_ROOT / "interface/scripts/validate-business-rag.mjs"
SPECIFICATION = PROJECT_ROOT / "data/rag-validation/business-cases.json"
RUN_LIVE = os.getenv("FOLIO_NOTEBOOK_LIVE", "0") == "1"  # Garder False pour les tests sans appel facturé.
print("Projet :", PROJECT_ROOT.name)
print("Mode live :", "activé" if RUN_LIVE else "désactivé")

Projet : MUTTI-AGENT
Mode live : activé


## 2. Cas métier et critères attendus

Les repères se trouvent dans `data/rag-validation/business-cases.json`. Les chiffres annuels de référence ont été relevés dans les rapports 10-K du corpus. Ils ne viennent pas de la connaissance générale du modèle. Le cas 2032 doit produire une absence de preuve.

Vous pouvez ajouter vos propres questions dans ce fichier en indiquant les entreprises, l’exercice, les rapports attendus et, si nécessaire, une valeur à retrouver dans les passages.

In [2]:
specification = json.loads(SPECIFICATION.read_text(encoding="utf-8"))
CASES = specification["cases"]

def table(rows, columns):
    def clean(value):
        if value is True: value = "oui"
        elif value is False: value = "non"
        elif value is None: value = "—"
        return html.escape(str(value))
    heading = "<tr>" + "".join("<th>" + clean(key) + "</th>" for key in columns) + "</tr>"
    body = "".join("<tr>" + "".join("<td>" + clean(row.get(key)) + "</td>" for key in columns) + "</tr>" for row in rows)
    display(HTML("<table><thead>" + heading + "</thead><tbody>" + body + "</tbody></table>"))

table([{"cas": c["id"], "entreprises": ", ".join(c["companies"]), "année": c["year"],
        "valeurs attendues": ", ".join(c["expectedValues"]) or "absence de preuve", "parcours": c["kind"]}
       for c in CASES], ["cas", "entreprises", "année", "valeurs attendues", "parcours"])

cas,entreprises,année,valeurs attendues,parcours
meta_revenue_2024,meta,2024,"164,501",chat
google_revenue_2024,google,2024,"350,018",chat
apple_sales_2024,apple,2024,"391,035",chat
amazon_sales_2024,amazon,2024,"637,959",chat
compare_meta_google_2024,"meta, google",2024,"164,501, 350,018",thematic
meta_future_abstention,meta,2032,absence de preuve,chat


In [3]:
def run_validation(mode="offline", case_ids=None):
    job = {"mode": mode, "caseIds": case_ids or [c["id"] for c in CASES]}
    if mode == "live":
        if not RUN_LIVE:
            raise RuntimeError("Activez RUN_LIVE pour autoriser les appels au fournisseur configuré.")
        job["allowPaidCalls"] = True
    completed = subprocess.run(
        [NODE, "--use-system-ca", str(RUNNER)], input=json.dumps(job, ensure_ascii=False),
        cwd=PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8", timeout=600,
    )
    if completed.returncode:
        raise RuntimeError(completed.stderr.strip() or "La validation n’a pas terminé.")
    return json.loads(completed.stdout)

def save_report(report):
    folder = PROJECT_ROOT / "interface/.runtime/validation-reports"
    folder.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target = folder / f"business-rag-{report['mode']}-{stamp}.json"
    target.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return target

## 3. Validation locale de la recherche — sans appel au modèle

Ce contrôle utilise le vrai catalogue et les textes déjà extraits des PDF. Il vérifie aussi l’isolation d’une note de mémoire contenant une instruction suspecte, dans une session de test séparée.

**Lecture des mesures** : le rappel documentaire mesure la présence des *rapports attendus* parmi les passages. La couverture des valeurs indique uniquement que les chiffres attendus sont retrouvés dans ces passages. Ce ne sont pas une précision sémantique annotée, un score de factualité ou une preuve de véracité des documents.

In [4]:
offline = run_validation()
print("Date du contrôle :", offline["at"])
print("Documents :", len(offline["catalog"]["documents"]))
table([{"entreprise": company, "documents": sum(d["company"] == company for d in offline["catalog"]["documents"])}
       for company in offline["catalog"]["companies"]], ["entreprise", "documents"])
table([{"cas": c["id"], "passages": c["retrieval"]["passageCount"], "périmètre correct": c["retrieval"]["scopeOk"],
        "rappel des rapports": c["retrieval"]["expectedSourceRecall"], "couverture valeurs": c["retrieval"]["expectedValueCoverage"],
        "budget respecté": c["retrieval"]["withinBudget"], "durée recherche (ms)": c["retrieval"]["durationMs"], "réussi": c["retrieval"]["pass"]}
       for c in offline["cases"]], ["cas", "passages", "périmètre correct", "rappel des rapports", "couverture valeurs", "budget respecté", "durée recherche (ms)", "réussi"])

Date du contrôle : 2026-11-15T17:24:27.429Z
Documents : 29


entreprise,documents
amazon,8
apple,6
google,6
meta,9


cas,passages,périmètre correct,rappel des rapports,couverture valeurs,budget respecté,durée recherche (ms),réussi
meta_revenue_2024,3,oui,1,1,oui,3191,oui
google_revenue_2024,3,oui,1,1,oui,54,oui
apple_sales_2024,3,oui,1,1,oui,40,oui
amazon_sales_2024,3,oui,1,1,oui,41,oui
compare_meta_google_2024,6,oui,1,1,oui,70,oui
meta_future_abstention,0,oui,—,—,oui,2,oui


In [5]:
table(offline["security"]["checks"], ["name", "passed"])
offline_report = save_report(offline)
print("Rapport local enregistré :", offline_report.relative_to(PROJECT_ROOT))
assert offline["allPassed"], "Un contrôle local a échoué : examiner les passages avant de poursuivre."

name,passed
instruction_documentaire_isolee,oui
source_inventee_rejetee,oui
chiffre_invente_rejete,oui
annee_source_incorrecte_rejetee,oui
shell_interdit,oui
redacteur_sans_recherche,oui
document_hors_perimetre,oui
memoire_suspecte_isolee_par_le_service,oui


Rapport local enregistré : interface\.runtime\validation-reports\business-rag-offline-20261115T172431Z.json


## 4. Examiner un passage et sa provenance

Un identifiant, un numéro de page quand l’extraction le permet et une empreinte du texte sont conservés. L’empreinte identifie le passage ; elle ne certifie ni l’authenticité du PDF ni la vérité de son contenu. Vérifiez dans le rapport d’origine l’exercice fiscal, les unités et la monnaie.

In [6]:
sample = next(c for c in offline["cases"] if c["id"] == "meta_revenue_2024")
table([{"rapport": p["name"], "page": p["page"], "année source": p["year"], "id": p["id"],
        "autorité d’instruction": p["instructionAuthority"]} for p in sample["passages"]],
      ["rapport", "page", "année source", "id", "autorité d’instruction"])
if sample["passages"]:
    print(sample["passages"][0]["text"][:900])

rapport,page,année source,id,autorité d’instruction
meta 10-k 2024,101,2024,evidence-5bc2b04beacc79d4a2b6,none
meta 10-k 2024,73,2024,evidence-ce59ee7693d229b398e9,none
meta 10-k 2024,74,2024,evidence-f3d80da22bcf8f3092b0,none


## Note 2. Revenue

Revenue disaggregated by revenue source and by segment consists of the following (in millions):

|                | Year Ended December 31,   | Year Ended December 31,   | Year Ended December 31,   |
|----------------|---------------------------|---------------------------|---------------------------|
|                | 2024                      | 2023                      | 2022                      |
| Advertising    | $ 160,633                 | $ 131,948                 | $ 113,642                 |
| Other revenue  | 1,722                     | 1,058                     | 808                       |
| Family of Apps | 162,355                   | 133,006                   | 114,450                   |
| Reality Labs   | 2,146                     | 1,896                     | 2,159                     |
| Total revenue  | $ 164,501                 | $ 134,


## 5. Validation réelle du chat et du résumé — optionnelle et potentiellement facturée

Avec `RUN_LIVE = True` (ou `FOLIO_NOTEBOOK_LIVE=1`), le serveur transmet au fournisseur configuré les questions, les passages et le contexte borné des sessions de test. La clé reste dans la configuration privée de l’application et n’est jamais affichée ici.

Par défaut, seuls trois cas passent en live : le revenu Meta, la comparaison Meta / Google et l’abstention 2032. Le cas sans passage doit renvoyer `NO_EVIDENCE`, sans réponse fabriquée : cet état constitue la réussite du test d’abstention. Le résumé exécute les quatre rôles ; le rédacteur sélectionne uniquement des affirmations déjà contrôlées.

Les instructions de documents et notes sont traitées comme données non fiables. Le filtrage est heuristique, les contrôles de cohérence par le modèle peuvent se tromper et cette validation ne garantit pas une protection universelle contre les injections.

In [7]:
LIVE_CASE_IDS = [c["id"] for c in CASES if c.get("liveDefault")]
live = run_validation("live", LIVE_CASE_IDS) if RUN_LIVE else None
if live:
    print("Modèle utilisé :", live["backend"]["model"])
    table([{"cas": c["id"], "état": c["generation"]["status"], "durée (s)": round(c["generation"]["durationMs"]/1000, 2),
            "citations": len(c["generation"]["citations"]), "citations exactes": c["generation"]["citationsExact"],
            "événements": c["generation"]["eventCount"], "réussi": c["generation"]["pass"]} for c in live["cases"]],
          ["cas", "état", "durée (s)", "citations", "citations exactes", "événements", "réussi"])
else:
    print("Live désactivé : les contrôles de recherche et de sécurité restent exécutés.")

Modèle utilisé : gemini-3.8-flash


cas,état,durée (s),citations,citations exactes,événements,réussi
meta_revenue_2024,completed,9.66,1,oui,4,oui
compare_meta_google_2024,completed,22.0,2,oui,8,oui
meta_future_abstention,failed,2.63,0,oui,2,oui


In [8]:
if live:
    for case in live["cases"]:
        print("\nCas :", case["id"])
        if case["generation"]["answer"]:
            print(case["generation"]["answer"])  # Texte non fiable affiché sans exécuter de HTML.
            table([{"rapport": c["title"], "page": c["page"], "extrait exact": c["quote"]}
                   for c in case["generation"]["citations"]], ["rapport", "page", "extrait exact"])
        else:
            print("Abstention / erreur contrôlée :", case["generation"]["error"]["code"])
    live_report = save_report(live)
    print("Rapport live enregistré :", live_report.relative_to(PROJECT_ROOT))
    assert live["allPassed"], "Au moins un contrôle live a échoué ; consulter le rapport."


Cas : meta_revenue_2024
Pour l'exercice clos le 31 décembre 2024, le chiffre d'affaires total annuel publié par Meta s'élève à 164 501 millions de dollars américains. [1]


rapport,page,extrait exact
meta 10-k 2024,101,"| Total revenue | $ 164,501 | $ 134,902 | $ 116,609 |"



Cas : compare_meta_google_2024
Pour l'exercice clos le 31 décembre 2024, le chiffre d'affaires total annuel publié de Google (Alphabet) s'est élevé à 350 018 millions de dollars américains. [1]

Pour l'exercice clos le 31 décembre 2024, le chiffre d'affaires total annuel publié de Meta s'est élevé à 164 501 millions de dollars américains. [2]


rapport,page,extrait exact
google 10-k 2024,37,"| Total revenues | $ 307,394 | $ 350,018 |"
meta 10-k 2024,101,"| Total revenue | $ 164,501 | $ 134,902 | $ 116,609 |"



Cas : meta_future_abstention
Abstention / erreur contrôlée : NO_EVIDENCE
Rapport live enregistré : interface\.runtime\validation-reports\business-rag-live-20261115T172533Z.json


## 6. Latence et limites de cette validation

- La première recherche locale inclut la construction de l’index. Les suivantes réutilisent cet index.
- En live, les passages ont déjà été préparés pour les contrôles documentaires : la durée du traitement est donc mesurée avec un **index chaud**. Elle inclut le worker Python, les appels au modèle et les validations, mais pas le réseau navigateur.
- Ces quelques requêtes ne suffisent pas pour annoncer un P95 de production. Refaire plusieurs séries représentatives, avec et sans cache, avant de dimensionner l’hébergement.
- Les assertions de citation vérifient un appui textuel, pas la vérité universelle de l’affirmation. Relire les unités, périodes, citations et contradictions éventuelles.
- Ce notebook ne remplace pas les tests E2E de navigation, les tests de connexion ou les tests utilisateurs. Les API du site doivent refuser une requête sans session ; ce point est testé séparément par `node interface/scripts/check-rag-api.mjs`.

### Critères de revue humaine

1. Le bon rapport et le bon exercice ont-ils été utilisés ?
2. La réponse respecte-t-elle les unités et la monnaie du tableau ?
3. Les affirmations sont-elles réellement appuyées par les passages cités ?
4. La comparaison garde-t-elle les entreprises et métriques distinctes ?
5. L’absence de source conduit-elle à une abstention explicite ?
6. Le temps d’attente et le suivi des agents sont-ils compréhensibles dans l’interface ?

Pour une évaluation statistique, ajouter un jeu de questions représentatif, des passages pertinents annotés et une revue indépendante. Ne pas renommer les présentes mesures en précision, MRR ou factualité sans ces annotations.